# EDA com Full Join das Abas

Este notebook carrega o Excel `BASE DE DADOS PEDE 2024 - DATATHON.xlsx`, faz um *full join* entre as abas `PEDE2022`, `PEDE2023`, `PEDE2024` pela chave `RA` e gera um EDA básico do resultado.

 INDE = Indice de desenvolvimento educacional uma media ponderada dos outros indicadores  
 IAN - Indicadores de adequação de nível{ (defasagem) em relação a fase atual}
 IDA - Indicador de desenvolvimento acadêmico {exames de avaliação interna}
 IEG - Indicador de Engajamento {Engajamento nas atividades curriculares ou voluntariado}
 IAA - Indicador de auto avaliação {autoavaliação de sentimento}
 IPS - Indicador Psicossocial {avaliação psico de interação social,emocional e comportamental}
 IPP - Indicador Psicopedagógico {avaliação de psico no aprendizado, cognitivo}
 IPV - Indicador do Ponto de Virada {avaliação psico de tipo de QI e engajamento}

# crianca 10 anos serie certa, indicadores relação de tipo essa crianca no fim do ano vai ter mais dificuldades 
+ dificuldade
+ atencao estudos

------
# todas as criancas 5 serie,   
algumas estao atrasadas ( + idade ) = + defasagem
notas mais baixas média dos indicadores = + defasagem { depedendo do indicador ++defasagem ou + defasagem ou ++++ defasagem}

combinacoes
idade mt atrasada(++defasagem), notas ruins(+defasagem), engajamentoruim(+defasagem)
idade mt atrasada(++defasagem), notas boas(+defasagem), engajamento ruim(+defasagem)
idade atrasada(+defasagem), notas boas(+defasagem), engajamento ruim(+defasagem)
idade ok(+defasagem), notas ruim(+defasagem), engajamento +/- (+defasagem)


------------- ideia ------------
base de dados (input)
ano nascimento, serie, notas. (+ indicadores de impacto, se tiver indicador x previsão é mais assertiva)

output
nota de risco de defasagem, grau de risco (nota >7 muito alto, 3 > nota < 7 risco medio, nota <3 risco baixo)


In [1]:
import pandas as pd

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

FILE_PATH = "/workspaces/Datathon-Machine-Learning-Engineering/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# ============================================================================
# 1. CARREGAR DADOS
# ============================================================================

print("📂 Carregando dados...")
df_2022 = pd.read_excel(FILE_PATH, sheet_name="PEDE2022")
df_2023 = pd.read_excel(FILE_PATH, sheet_name="PEDE2023")
df_2024 = pd.read_excel(FILE_PATH, sheet_name="PEDE2024")

print(f"  ✓ 2022: {df_2022.shape[0]:,} linhas x {df_2022.shape[1]} colunas")
print(f"  ✓ 2023: {df_2023.shape[0]:,} linhas x {df_2023.shape[1]} colunas")
print(f"  ✓ 2024: {df_2024.shape[0]:,} linhas x {df_2024.shape[1]} colunas")

📂 Carregando dados...
  ✓ 2022: 860 linhas x 42 colunas
  ✓ 2023: 1,014 linhas x 48 colunas
  ✓ 2024: 1,156 linhas x 50 colunas


In [2]:
COLUNAS_SEM_SUFIXO = ['RA']

# Função para adicionar sufixo nas colunas
def add_suffix_to_columns(df, suffix, exclude_cols):
    """
    Adiciona sufixo às colunas, exceto as que estão na lista de exclusão
    """
    new_columns = {}
    for col in df.columns:
        if col in exclude_cols:
            new_columns[col] = col  # Mantém o nome original
        else:
            new_columns[col] = f"{col}_{suffix}"  # Adiciona sufixo
    
    return df.rename(columns=new_columns)

# Renomeia cada base
df_2022_renamed = add_suffix_to_columns(df_2022, '2022', COLUNAS_SEM_SUFIXO)
df_2023_renamed = add_suffix_to_columns(df_2023, '2023', COLUNAS_SEM_SUFIXO)
df_2024_renamed = add_suffix_to_columns(df_2024, '2024', COLUNAS_SEM_SUFIXO)

print(df_2022_renamed.columns.tolist())
print(df_2023_renamed.columns.tolist())
print(df_2024_renamed.columns.tolist())

['RA', 'Fase_2022', 'Turma_2022', 'Nome_2022', 'Ano nasc_2022', 'Idade 22_2022', 'Gênero_2022', 'Ano ingresso_2022', 'Instituição de ensino_2022', 'Pedra 20_2022', 'Pedra 21_2022', 'Pedra 22_2022', 'INDE 22_2022', 'Cg_2022', 'Cf_2022', 'Ct_2022', 'Nº Av_2022', 'Avaliador1_2022', 'Rec Av1_2022', 'Avaliador2_2022', 'Rec Av2_2022', 'Avaliador3_2022', 'Rec Av3_2022', 'Avaliador4_2022', 'Rec Av4_2022', 'IAA_2022', 'IEG_2022', 'IPS_2022', 'Rec Psicologia_2022', 'IDA_2022', 'Matem_2022', 'Portug_2022', 'Inglês_2022', 'Indicado_2022', 'Atingiu PV_2022', 'IPV_2022', 'IAN_2022', 'Fase ideal_2022', 'Defas_2022', 'Destaque IEG_2022', 'Destaque IDA_2022', 'Destaque IPV_2022']
['RA', 'Fase_2023', 'INDE 2023_2023', 'Pedra 2023_2023', 'Turma_2023', 'Nome Anonimizado_2023', 'Data de Nasc_2023', 'Idade_2023', 'Gênero_2023', 'Ano ingresso_2023', 'Instituição de ensino_2023', 'Pedra 20_2023', 'Pedra 21_2023', 'Pedra 22_2023', 'Pedra 23_2023', 'INDE 22_2023', 'INDE 23_2023', 'Cg_2023', 'Cf_2023', 'Ct_2023'

In [3]:
# Padronizacao das fases (simples, coluna por coluna)
FASE_MAP = {
    "0": "ALFA",
    "1": "FASE 1",
    "2": "FASE 2",
    "3": "FASE 3",
    "4": "FASE 4",
    "5": "FASE 5",
    "6": "FASE 6",
    "7": "FASE 7",
    "8": "FASE 8",
    "9": "FASE 9",
}

# Fase 2022
fase_2022_base = df_2022_renamed["Fase_2022"].astype("string").str.strip().str.upper()
fase_2022_digit = fase_2022_base.str.extract(r"(\d)", expand=False)
df_2022_renamed["Fase_2022_adj"] = fase_2022_digit.map(FASE_MAP).fillna(fase_2022_base)

# Fase 2023 (ja correta, mas padroniza caixa/espacos)
fase_2023_base = df_2023_renamed["Fase_2023"].astype("string").str.strip().str.upper()
fase_2023_digit = fase_2023_base.str.extract(r"(\d)", expand=False)
df_2023_renamed["Fase_2023_adj"] = fase_2023_digit.map(FASE_MAP).fillna(fase_2023_base)

# Fase 2024 (mistura numero e texto -> pega o primeiro digito)
fase_2024_base = df_2024_renamed["Fase_2024"].astype("string").str.strip().str.upper()
fase_2024_digit = fase_2024_base.str.extract(r"(\d)", expand=False)
df_2024_renamed["Fase_2024_adj"] = fase_2024_digit.map(FASE_MAP).fillna(fase_2024_base)



In [4]:
print(sorted(map(int, df_2022_renamed["Defas_2022"].dropna().unique())))
print(sorted(map(int, df_2023_renamed["Defasagem_2023"].dropna().unique())))
print(sorted(map(int, df_2024_renamed["Defasagem_2024"].dropna().unique())))

[-5, -4, -3, -2, -1, 0, 1, 2]
[-4, -3, -2, -1, 0, 1, 2]
[-3, -2, -1, 0, 1, 2, 3]


In [5]:
# 4. CORRELAÇÃO COM DEFAS_2022 (PRÉ-MERGE, BASE 2022)

base = df_2022_renamed.copy()
target_col = "Defas_2022"

# Mantém apenas colunas numéricas (convertendo quando possível)
numeric_df = base.apply(lambda s: pd.to_numeric(s, errors="coerce"))

if target_col not in numeric_df.columns:
    raise ValueError(f"A coluna alvo {target_col} não está disponível para cálculo numérico.")

# Correlação de Spearman (mais robusta para relações monotônicas)
corr_target = numeric_df.corr(method="spearman")[target_col].dropna()
corr_target = corr_target.drop(index=target_col, errors="ignore")

top_corr = (
    corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)
    .head(20)
    .to_frame(name="corr_spearman")
    .reset_index()
    .rename(columns={"index": "variavel"})
)

print("\n📊 Top variáveis mais correlacionadas com", target_col)
display(top_corr)


📊 Top variáveis mais correlacionadas com Defas_2022


,variavel,corr_spearman
0,IAN_2022,0.886585
1,INDE 22_2022,0.416497
2,Cg_2022,-0.416471
3,Cf_2022,-0.388717
4,Idade 22_2022,-0.326028
5,Ano nasc_2022,0.326028
6,Ct_2022,-0.323271
7,Inglês_2022,0.232317
8,IEG_2022,0.199860
9,IPV_2022,0.153828


In [6]:
# 5. MÉDIAS POR FASE COM TODAS AS VARIÁVEIS (BASE 2022)
base_2022 = df_2022_renamed.copy()

# Colunas a excluir do cálculo de média
excluir = ["RA", "Fase_2022", "Fase_2022_adj"]

# Converte para numérico quando possível
for col in base_2022.columns:
    if col not in excluir:
        base_2022[col] = pd.to_numeric(base_2022[col], errors="coerce")

# Variáveis numéricas válidas
cols_numericas = [
    c for c in base_2022.columns
    if c not in excluir and base_2022[c].notna().any() and pd.api.types.is_numeric_dtype(base_2022[c])
]

# Apenas média por fase
medias_por_fase_todas = (
    base_2022.groupby("Fase_2022_adj")[cols_numericas]
    .mean()
    .round(2)
    .reset_index()
    .sort_values("Fase_2022_adj")
)

print(f"Total de variáveis numéricas consideradas: {len(cols_numericas)}")
display(medias_por_fase_todas)

Total de variáveis numéricas consideradas: 18


,Fase_2022_adj,Ano nasc_2022,Idade 22_2022,Ano ingresso_2022,INDE 22_2022,Cg_2022,Cf_2022,Ct_2022,Nº Av_2022,IAA_2022,IEG_2022,IPS_2022,IDA_2022,Matem_2022,Portug_2022,Inglês_2022,IPV_2022,IAN_2022,Defas_2022
0,ALFA,2013.07,8.93,2021.66,7.37,348.13,95.5,5.12,2.00,8.98,8.09,7.01,7.14,7.40,6.86,NaN,7.56,6.80,-0.92
1,FASE 1,2011.37,10.63,2020.80,7.20,392.58,96.5,6.78,2.76,8.64,8.52,7.08,6.46,5.91,6.99,NaN,7.36,5.74,-1.07
2,FASE 2,2009.91,12.09,2020.33,6.96,464.43,78.0,7.21,3.00,8.41,8.17,6.82,5.41,4.67,6.12,NaN,7.34,6.40,-0.83
3,FASE 3,2008.20,13.80,2020.27,6.60,520.86,74.5,7.22,3.91,7.49,7.07,6.72,5.14,4.91,5.63,5.13,6.55,6.99,-0.91
4,FASE 4,2007.07,14.93,2019.05,7.01,439.32,38.5,7.29,4.00,7.71,7.66,6.61,6.05,5.38,6.49,6.31,7.21,6.48,-0.96
5,FASE 5,2005.95,16.05,2019.47,6.88,472.83,30.5,6.05,3.55,8.11,7.34,6.86,5.87,5.54,5.81,6.42,7.26,6.25,-1.05
6,FASE 6,2005.17,16.83,2019.22,7.20,374.78,9.5,9.50,4.00,6.51,7.03,7.96,6.69,7.81,4.41,7.86,8.22,5.83,-0.83
7,FASE 7,2003.67,18.33,2019.33,6.64,530.71,11.0,6.05,4.00,7.01,7.24,6.52,5.25,5.58,4.09,5.90,7.18,6.19,-0.76


In [7]:
# 7. MÉDIAS DAS VARIÁVEIS POR FASE_2022_adj E VALOR DE DEFAS_2022
base_2022 = df_2022_renamed.copy()

# Converter Defas para número e remover linhas sem fase/defas
base_2022["Defas_2022"] = pd.to_numeric(base_2022["Defas_2022"], errors="coerce")
base_2022 = base_2022.dropna(subset=["Fase_2022_adj", "Defas_2022"])

# Definir variáveis numéricas para média
excluir = ["RA", "Fase_2022", "Fase_2022_adj", "Defas_2022"]
for col in base_2022.columns:
    if col not in excluir:
        base_2022[col] = pd.to_numeric(base_2022[col], errors="coerce")

cols_numericas = [
    c for c in base_2022.columns
    if c not in excluir and pd.api.types.is_numeric_dtype(base_2022[c]) and base_2022[c].notna().any()
]

medias_fase_defas = (
    base_2022.groupby(["Fase_2022_adj", "Defas_2022"])[cols_numericas]
    .mean()
    .round(2)
    .reset_index()
    .sort_values(["Fase_2022_adj", "Defas_2022"])
)

print(f"Total de variáveis numéricas consideradas: {len(cols_numericas)}")
display(medias_fase_defas)

Total de variáveis numéricas consideradas: 17


,Fase_2022_adj,Defas_2022,Ano nasc_2022,Idade 22_2022,Ano ingresso_2022,INDE 22_2022,Cg_2022,Cf_2022,Ct_2022,Nº Av_2022,IAA_2022,IEG_2022,IPS_2022,IDA_2022,Matem_2022,Portug_2022,Inglês_2022,IPV_2022,IAN_2022
0,ALFA,-3,2010.00,12.00,2022.00,7.28,402.00,118.00,10.00,2.00,10.00,9.70,7.50,6.70,7.30,6.00,NaN,7.08,2.5
1,ALFA,-2,2011.73,10.27,2021.17,7.02,449.87,121.65,6.08,2.00,9.09,7.84,6.94,6.67,7.00,6.32,NaN,7.48,5.0
2,ALFA,-1,2013.00,9.00,2021.78,7.32,364.69,101.12,5.43,2.00,8.78,8.30,6.88,7.48,7.71,7.21,NaN,7.74,5.0
3,ALFA,0,2014.20,7.80,2021.90,7.68,254.36,69.93,4.01,2.00,9.09,8.05,7.17,7.17,7.41,6.92,NaN,7.44,10.0
4,FASE 1,-3,2008.00,14.00,2020.67,6.09,654.67,154.33,8.67,2.33,9.00,8.30,6.27,5.03,2.83,7.17,NaN,5.81,2.5
5,FASE 1,-2,2009.77,12.23,2019.92,6.91,466.64,113.64,7.44,2.74,8.24,7.90,7.01,6.47,6.25,6.66,NaN,7.18,5.0
6,FASE 1,-1,2011.55,10.45,2020.97,7.15,410.65,101.44,7.18,2.72,8.68,8.53,7.08,6.47,5.92,6.99,NaN,7.45,5.0
7,FASE 1,0,2013.00,9.00,2021.29,7.88,201.50,49.54,4.11,2.96,8.96,9.30,7.28,6.59,5.72,7.44,NaN,7.41,10.0
8,FASE 1,1,2014.00,8.00,2021.00,8.00,146.50,36.50,4.50,3.00,9.25,9.70,7.50,6.55,6.00,7.10,NaN,7.47,10.0
9,FASE 2,-4,2006.00,16.00,2018.00,5.52,790.00,148.00,7.00,3.00,10.00,5.40,5.00,4.60,4.70,4.50,NaN,6.78,2.5


In [8]:
# 8. (2023) MÉDIAS POR FASE + MÉDIAS POR FASE E DEFASAGEM
base_2023 = df_2023_renamed.copy()

# Parte A: médias por fase com todas as variáveis numéricas
excluir_2023 = ["RA", "Fase_2023", "Fase_2023_adj"]
for col in base_2023.columns:
    if col not in excluir_2023:
        base_2023[col] = pd.to_numeric(base_2023[col], errors="coerce")

cols_numericas_2023 = [
    c for c in base_2023.columns
    if c not in excluir_2023 and pd.api.types.is_numeric_dtype(base_2023[c]) and base_2023[c].notna().any()
]

medias_por_fase_todas_2023 = (
    base_2023.groupby("Fase_2023_adj")[cols_numericas_2023]
    .mean()
    .round(2)
    .reset_index()
    .sort_values("Fase_2023_adj")
)

print(f"[2023] Variáveis numéricas consideradas (fase): {len(cols_numericas_2023)}")
display(medias_por_fase_todas_2023)

# Parte B: médias por fase e por valor de defasagem
base_2023_def = base_2023.copy()
base_2023_def["Defasagem_2023"] = pd.to_numeric(base_2023_def["Defasagem_2023"], errors="coerce")
base_2023_def = base_2023_def.dropna(subset=["Fase_2023_adj", "Defasagem_2023"])

excluir_def_2023 = ["RA", "Fase_2023", "Fase_2023_adj", "Defasagem_2023"]
cols_numericas_def_2023 = [
    c for c in base_2023_def.columns
    if c not in excluir_def_2023 and pd.api.types.is_numeric_dtype(base_2023_def[c]) and base_2023_def[c].notna().any()
]

medias_fase_defas_2023 = (
    base_2023_def.groupby(["Fase_2023_adj", "Defasagem_2023"])[cols_numericas_def_2023]
    .mean()
    .round(2)
    .reset_index()
    .sort_values(["Fase_2023_adj", "Defasagem_2023"])
)

print(f"[2023] Variáveis numéricas consideradas (fase + defasagem): {len(cols_numericas_def_2023)}")
display(medias_fase_defas_2023)

[2023] Variáveis numéricas consideradas (fase): 16


,Fase_2023_adj,INDE 2023_2023,Idade_2023,Ano ingresso_2023,INDE 22_2023,Nº Av_2023,IAA_2023,IEG_2023,IPS_2023,IPP_2023,IDA_2023,Mat_2023,Por_2023,Ing_2023,IPV_2023,IAN_2023,Defasagem_2023
0,ALFA,7.68,8.69,2022.61,7.47,2.0,7.46,8.94,5.78,6.78,7.42,7.43,7.36,NaN,8.32,7.46,-0.72
1,FASE 1,7.45,10.25,2021.93,7.36,3.0,7.20,8.74,6.12,7.77,6.81,6.10,7.47,NaN,8.10,6.17,-0.84
2,FASE 2,7.37,11.81,2021.48,7.29,3.0,6.62,8.80,4.53,8.14,6.74,5.83,7.59,NaN,8.21,6.88,-0.70
3,FASE 3,6.97,13.45,2020.67,7.01,4.0,6.38,8.44,4.67,7.53,5.75,6.16,5.84,5.24,7.58,7.58,-0.61
4,FASE 4,7.07,14.53,2020.47,7.14,4.0,6.95,8.35,3.79,7.79,6.00,6.04,5.81,6.16,7.95,7.47,-0.62
5,FASE 5,6.87,15.76,2020.09,7.32,4.0,6.44,8.45,4.35,7.56,5.90,5.66,4.79,7.24,7.45,6.73,-0.80
6,FASE 6,7.24,16.81,2019.70,7.15,4.0,6.38,8.50,5.80,7.81,6.81,7.41,5.55,7.47,7.73,6.36,-0.94
7,FASE 7,7.83,16.79,2020.09,8.03,4.0,6.51,9.17,4.96,7.84,7.81,8.04,7.11,8.31,7.88,8.70,0.30
8,FASE 8,NaN,19.56,2020.57,6.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.00,0.03


[2023] Variáveis numéricas consideradas (fase + defasagem): 15


,Fase_2023_adj,Defasagem_2023,INDE 2023_2023,Idade_2023,Ano ingresso_2023,INDE 22_2023,Nº Av_2023,IAA_2023,IEG_2023,IPS_2023,IPP_2023,IDA_2023,Mat_2023,Por_2023,Ing_2023,IPV_2023,IAN_2023
0,ALFA,-3,5.97,12.00,2023.00,NaN,2.0,10.00,9.30,2.52,5.31,3.90,4.50,3.30,NaN,6.50,2.5
1,ALFA,-2,7.05,10.26,2022.13,7.06,2.0,7.21,8.54,4.97,6.46,6.96,7.04,6.84,NaN,7.97,5.0
2,ALFA,-1,7.50,9.00,2022.49,7.70,2.0,7.29,9.08,5.84,6.78,7.48,7.54,7.36,NaN,8.51,5.0
3,ALFA,0,8.06,7.81,2022.87,7.79,2.0,7.64,9.02,6.10,6.92,7.61,7.55,7.62,NaN,8.36,10.0
4,FASE 1,-3,6.49,NaN,2023.00,NaN,3.0,8.50,9.80,7.52,6.41,3.20,2.30,4.00,NaN,7.06,2.5
5,FASE 1,-2,6.68,12.12,2021.07,6.80,3.0,5.11,7.85,5.64,7.39,6.31,5.78,6.76,NaN,7.72,5.0
6,FASE 1,-1,7.43,10.44,2021.91,7.35,3.0,7.66,8.78,6.24,7.79,6.93,6.31,7.49,NaN,8.11,5.0
7,FASE 1,0,7.81,9.00,2022.23,7.72,3.0,6.44,8.90,5.90,7.90,6.83,5.85,7.75,NaN,8.24,10.0
8,FASE 1,1,7.59,8.00,2022.50,7.48,3.0,8.75,8.75,6.27,7.50,5.25,3.00,7.40,NaN,7.72,10.0
9,FASE 2,-3,6.79,NaN,2020.00,6.42,3.0,7.40,8.65,5.00,7.29,6.80,7.10,6.50,NaN,7.40,2.5


In [9]:
# 9. (2024) MÉDIAS POR FASE + MÉDIAS POR FASE E DEFASAGEM
base_2024 = df_2024_renamed.copy()

# Parte A: médias por fase com todas as variáveis numéricas
excluir_2024 = ["RA", "Fase_2024", "Fase_2024_adj"]
for col in base_2024.columns:
    if col not in excluir_2024:
        base_2024[col] = pd.to_numeric(base_2024[col], errors="coerce")

cols_numericas_2024 = [
    c for c in base_2024.columns
    if c not in excluir_2024 and pd.api.types.is_numeric_dtype(base_2024[c]) and base_2024[c].notna().any()
]

medias_por_fase_todas_2024 = (
    base_2024.groupby("Fase_2024_adj")[cols_numericas_2024]
    .mean()
    .round(2)
    .reset_index()
    .sort_values("Fase_2024_adj")
)

print(f"[2024] Variáveis numéricas consideradas (fase): {len(cols_numericas_2024)}")
display(medias_por_fase_todas_2024)

# Parte B: médias por fase e por valor de defasagem
base_2024_def = base_2024.copy()
base_2024_def["Defasagem_2024"] = pd.to_numeric(base_2024_def["Defasagem_2024"], errors="coerce")
base_2024_def = base_2024_def.dropna(subset=["Fase_2024_adj", "Defasagem_2024"])

excluir_def_2024 = ["RA", "Fase_2024", "Fase_2024_adj", "Defasagem_2024"]
cols_numericas_def_2024 = [
    c for c in base_2024_def.columns
    if c not in excluir_def_2024 and pd.api.types.is_numeric_dtype(base_2024_def[c]) and base_2024_def[c].notna().any()
]

medias_fase_defas_2024 = (
    base_2024_def.groupby(["Fase_2024_adj", "Defasagem_2024"])[cols_numericas_def_2024]
    .mean()
    .round(2)
    .reset_index()
    .sort_values(["Fase_2024_adj", "Defasagem_2024"])
)

print(f"[2024] Variáveis numéricas consideradas (fase + defasagem): {len(cols_numericas_def_2024)}")
display(medias_fase_defas_2024)

[2024] Variáveis numéricas consideradas (fase): 19


,Fase_2024_adj,INDE 2024_2024,Turma_2024,Data de Nasc_2024,Idade_2024,Ano ingresso_2024,INDE 22_2024,INDE 23_2024,Nº Av_2024,IAA_2024,IEG_2024,IPS_2024,IPP_2024,IDA_2024,Mat_2024,Por_2024,Ing_2024,IPV_2024,IAN_2024,Defasagem_2024
0,ALFA,7.50,NaN,1.441024e+15,8.69,2023.54,7.35,7.75,2.33,8.99,8.50,6.26,7.32,7.32,7.85,6.79,NaN,7.28,6.25,-0.87
1,FASE 1,7.56,NaN,1.386693e+15,10.39,2022.95,7.39,7.53,2.70,8.81,8.49,6.97,7.84,6.79,6.77,6.81,NaN,7.50,6.46,-0.79
2,FASE 2,7.37,NaN,1.348709e+15,11.62,2022.59,7.41,7.54,3.24,8.20,7.96,6.96,7.57,6.25,5.81,6.71,NaN,7.21,8.14,-0.35
3,FASE 3,7.11,NaN,1.298207e+15,13.21,2022.24,7.32,7.36,3.80,8.48,7.39,7.17,7.36,5.35,5.37,5.02,5.70,6.95,8.67,-0.15
4,FASE 4,7.30,NaN,1.244228e+15,14.93,2022.39,7.07,7.21,4.34,8.11,7.95,6.66,7.66,5.88,4.98,6.07,6.78,7.84,7.20,-0.52
5,FASE 5,7.47,NaN,1.211999e+15,15.87,2022.21,7.34,7.25,3.96,8.30,8.47,6.74,7.54,6.45,6.09,5.60,7.83,7.56,7.20,-0.58
6,FASE 6,7.84,NaN,1.180625e+15,16.96,2022.52,7.50,7.07,5.00,8.65,8.80,6.86,7.76,7.23,7.07,6.08,8.54,7.76,7.60,-0.48
7,FASE 7,7.58,NaN,1.194420e+15,16.51,2021.14,8.13,7.75,0.97,8.86,7.44,7.30,7.84,5.81,5.55,5.53,6.36,7.64,10.00,0.92
8,FASE 8,NaN,NaN,1.081410e+15,20.03,2021.30,7.23,7.79,0.00,NaN,0.00,NaN,NaN,8.00,7.00,7.00,10.00,NaN,10.00,0.00
9,FASE 9,NaN,9.0,1.025635e+15,21.82,2021.03,NaN,NaN,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.00,1.00


[2024] Variáveis numéricas consideradas (fase + defasagem): 18


,Fase_2024_adj,Defasagem_2024,INDE 2024_2024,Turma_2024,Data de Nasc_2024,Idade_2024,Ano ingresso_2024,INDE 22_2024,INDE 23_2024,Nº Av_2024,IAA_2024,IEG_2024,IPS_2024,IPP_2024,IDA_2024,Mat_2024,Por_2024,Ing_2024,IPV_2024,IAN_2024
0,ALFA,-2,7.18,NaN,1.392919e+15,10.12,2023.17,7.07,7.21,2.29,9.00,8.22,6.86,7.49,6.60,7.67,5.54,NaN,6.90,5.0
1,ALFA,-1,7.37,NaN,1.435934e+15,8.85,2023.43,7.55,7.88,2.32,8.95,8.46,6.25,7.32,7.35,7.90,6.79,NaN,7.31,5.0
2,ALFA,0,7.98,NaN,1.477362e+15,7.59,2024.00,NaN,NaN,2.37,9.09,8.74,5.98,7.24,7.60,7.83,7.38,NaN,7.41,10.0
3,FASE 1,-2,6.59,NaN,1.325381e+15,12.25,2022.94,7.02,6.38,2.62,7.91,7.30,6.57,7.14,5.95,5.81,6.09,NaN,6.37,5.0
4,FASE 1,-1,7.36,NaN,1.379358e+15,10.64,2022.93,7.24,7.37,2.69,8.80,8.38,6.94,7.85,6.67,6.71,6.63,NaN,7.47,5.0
5,FASE 1,0,8.28,NaN,1.420482e+15,9.31,2022.98,7.98,8.09,2.74,9.09,9.08,7.14,8.02,7.30,7.18,7.42,NaN,7.91,10.0
6,FASE 2,-2,5.97,NaN,1.277683e+15,14.00,2022.00,7.28,7.10,4.00,10.00,6.42,6.26,6.56,3.75,4.00,3.50,NaN,5.75,5.0
7,FASE 2,-1,6.61,NaN,1.319681e+15,12.47,2022.68,7.00,7.07,3.32,8.02,7.56,6.74,7.11,5.49,4.95,6.04,NaN,6.60,5.0
8,FASE 2,0,7.80,NaN,1.364714e+15,11.14,2022.54,7.61,7.75,3.18,8.25,8.20,7.10,7.83,6.66,6.27,7.04,NaN,7.55,10.0
9,FASE 2,1,8.51,NaN,1.402393e+15,10.00,2022.60,7.95,8.11,3.20,9.20,8.54,7.26,8.26,8.15,7.10,9.20,NaN,8.48,10.0


In [10]:

# Primeiro merge: 2022 + 2023
merged = df_2022_renamed.merge(
    df_2023_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)


# Segundo merge: (2022+2023) + 2024
merged = merged.merge(
    df_2024_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)

print(merged.shape)


(1661, 141)


In [11]:
total_alunos = merged['RA'].nunique()
print(f"  • Total de alunos únicos (RA): {total_alunos:,}")

# Alunos por ano
alunos_2022 = df_2022_renamed['RA'].nunique()
alunos_2023 = df_2023_renamed['RA'].nunique()
alunos_2024 = df_2024_renamed['RA'].nunique()

print(f"\n  • Alunos em 2022: {alunos_2022:,}")
print(f"  • Alunos em 2023: {alunos_2023:,}")
print(f"  • Alunos em 2024: {alunos_2024:,}")

# Alunos que aparecem em todos os anos
alunos_2022_2023 = set(df_2022_renamed['RA']) & set(df_2023_renamed['RA'])
alunos_2023_2024 = set(df_2023_renamed['RA']) & set(df_2024_renamed['RA'])
alunos_todos_anos = alunos_2022_2023 & set(df_2024_renamed['RA'])

print(f"\n  • Alunos em 2022 E 2023: {len(alunos_2022_2023):,}")
print(f"  • Alunos em 2023 E 2024: {len(alunos_2023_2024):,}")
print(f"  • Alunos todos anos: {len(alunos_todos_anos):,}")


  • Total de alunos únicos (RA): 1,661

  • Alunos em 2022: 860
  • Alunos em 2023: 1,014
  • Alunos em 2024: 1,156

  • Alunos em 2022 E 2023: 600
  • Alunos em 2023 E 2024: 765
  • Alunos todos anos: 468


In [12]:
print(merged.head())

print("\n📊 Info do dataset:")
print(f"  • Shape: {merged.shape}")
print(f"  • Colunas: {merged.shape[1]}")
print(f"  • Memória: {merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

        RA  Fase_2022 Turma_2022  Nome_2022  Ano nasc_2022  Idade 22_2022  \
0     RA-1        7.0          A    Aluno-1         2003.0           19.0   
1    RA-10        7.0          A   Aluno-10         2004.0           18.0   
2   RA-100        4.0          A  Aluno-100         2009.0           13.0   
3  RA-1000        NaN        NaN        NaN            NaN            NaN   
4  RA-1001        NaN        NaN        NaN            NaN            NaN   

  Gênero_2022  Ano ingresso_2022 Instituição de ensino_2022 Pedra 20_2022  \
0      Menina             2016.0             Escola Pública      Ametista   
1      Menina             2021.0             Escola Pública           NaN   
2      Menina             2019.0               Rede Decisão      Ametista   
3         NaN                NaN                        NaN           NaN   
4         NaN                NaN                        NaN           NaN   

   ... IAN_2024          Fase Ideal_2024  Defasagem_2024  Destaque IEG_202

In [13]:
# Valores unicos por coluna (brutos)
cols_fase = ["Fase_2022", "Fase_2023", "Fase_2024","Fase_2022_adj", "Fase_2023_adj", "Fase_2024_adj"]
for col in cols_fase:
    if col not in merged.columns:
        print(f"Coluna ausente: {col}")
        continue
    valores = merged[col].dropna().astype("string")
    unicos = sorted(valores.unique())
    print(f"\n{col} -> {len(unicos)} valores unicos (bruto):")
    print(unicos)


Fase_2022 -> 8 valores unicos (bruto):
['0.0', '1.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0']

Fase_2023 -> 9 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8']

Fase_2024 -> 72 valores unicos (bruto):
['1A', '1B', '1C', '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R', '2A', '2B', '2C', '2D', '2G', '2H', '2I', '2K', '2L', '2M', '2N', '2P', '2R', '2U', '3A', '3B', '3C', '3D', '3F', '3G', '3H', '3I', '3K', '3L', '3M', '3N', '3P', '3R', '3U', '4A', '4B', '4C', '4F', '4H', '4L', '4M', '4N', '4R', '5A', '5B', '5C', '5D', '5F', '5G', '5L', '5M', '5N', '6A', '6L', '7A', '7E', '8A', '8B', '8D', '8E', '8F', '9', 'ALFA']

Fase_2022_adj -> 8 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7']

Fase_2023_adj -> 9 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8']

Fase_2024_adj -> 10 valores unicos (bruto)

In [15]:
# Tabela de fase x serie ideal x idade x ano nascimento (ano atual)

from datetime import datetime

ano_ref = datetime.now().year

rows = [
    {"FASE": "ALFA", "FASE_SERIE": "ALFA (1° e 2° ano)", "SERIE": "1° ano", "IDADE": 7},
    {"FASE": "ALFA", "FASE_SERIE": "ALFA (1° e 2° ano)", "SERIE": "2° ano", "IDADE": 8},
    {"FASE": "FASE 1", "FASE_SERIE": "Fase 1 (3° e 4° ano)", "SERIE": "3° ano", "IDADE": 9},
    {"FASE": "FASE 1", "FASE_SERIE": "Fase 1 (3° e 4° ano)", "SERIE": "4° ano", "IDADE": 10},
    {"FASE": "FASE 2", "FASE_SERIE": "Fase 2 (5° e 6° ano)", "SERIE": "5° ano", "IDADE": 11},
    {"FASE": "FASE 2", "FASE_SERIE": "Fase 2 (5° e 6° ano)", "SERIE": "6° ano", "IDADE": 12},
    {"FASE": "FASE 3", "FASE_SERIE": "Fase 3 (7° e 8° ano)", "SERIE": "7° ano", "IDADE": 13},
    {"FASE": "FASE 3", "FASE_SERIE": "Fase 3 (7° e 8° ano)", "SERIE": "8° ano", "IDADE": 14},
    {"FASE": "FASE 4", "FASE_SERIE": "Fase 4 (9° ano)", "SERIE": "9° ano", "IDADE": 15},
    {"FASE": "FASE 5", "FASE_SERIE": "Fase 5 (1° EM)", "SERIE": "1° EM", "IDADE": 16},
    {"FASE": "FASE 6", "FASE_SERIE": "Fase 6 (2° EM)", "SERIE": "2° EM", "IDADE": 17},
    {"FASE": "FASE 7", "FASE_SERIE": "Fase 7 (3° EM)", "SERIE": "3° EM", "IDADE": 18},
    {"FASE": "FASE 8", "FASE_SERIE": "Fase 8 (Universitários)", "SERIE": "Universidade", "IDADE": "18+"},
]

fase_serie_ideal = pd.DataFrame(rows)

fase_serie_ideal["ANO NASCIMENTO"] = fase_serie_ideal["IDADE"].apply(
    lambda idade: f"{ano_ref - 18}-" if idade == "18+" else str(ano_ref - int(idade))
)

fase_serie_ideal["IDADE"] = fase_serie_ideal["IDADE"].astype(str)

fase_serie_ideal = fase_serie_ideal[[
    "FASE",
    "FASE_SERIE",
    "SERIE",
    "IDADE",
    "ANO NASCIMENTO",
]]

display(fase_serie_ideal)


,FASE,FASE_SERIE,SERIE,IDADE,ANO NASCIMENTO
0,ALFA,ALFA (1° e 2° ano),1° ano,7,2019
1,ALFA,ALFA (1° e 2° ano),2° ano,8,2018
2,FASE 1,Fase 1 (3° e 4° ano),3° ano,9,2017
3,FASE 1,Fase 1 (3° e 4° ano),4° ano,10,2016
4,FASE 2,Fase 2 (5° e 6° ano),5° ano,11,2015
5,FASE 2,Fase 2 (5° e 6° ano),6° ano,12,2014
6,FASE 3,Fase 3 (7° e 8° ano),7° ano,13,2013
7,FASE 3,Fase 3 (7° e 8° ano),8° ano,14,2012
8,FASE 4,Fase 4 (9° ano),9° ano,15,2011
9,FASE 5,Fase 5 (1° EM),1° EM,16,2010
